# 第06章：vLLM 矩阵乘法算子全景

## 本章内容

深入分析 vLLM 中用于 **GQA (Grouped Query Attention)**、**MLA (Multi-head Latent Attention)**、
**GDN (GatedDeltaNet)** 和 **MLP/MoE** 的矩阵乘法算子：

1. LLM 推理中矩阵乘法出现在哪里？
2. 每种场景使用了哪些算子实现？
3. Triton vs cuBLAS vs FlashAttention vs CUTLASS 的选择逻辑
4. 哪些是性能最关键的矩阵乘法？

## 前置知识
- Notebook 00-05 的 Triton 基础
- 矩阵乘法 (GEMM) 的基本概念
- Transformer 注意力机制的基本原理

## 6.1 LLM 中的矩阵乘法全景

一个 Transformer Decoder Layer 中，**所有计算密集型操作都是矩阵乘法**。

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Transformer Decoder Layer                       │
│                                                                     │
│  ┌─────────────────── Attention Block ──────────────────────────┐  │
│  │                                                               │  │
│  │  hidden_states ──┬──▸ Q_proj (GEMM①)                        │  │
│  │                  ├──▸ K_proj (GEMM②)     ▸ Q*Kᵀ (GEMM④)    │  │
│  │                  └──▸ V_proj (GEMM③)     ▸ attn*V (GEMM⑤)  │  │
│  │                                           ▸ O_proj (GEMM⑥)  │  │
│  └───────────────────────────────────────────────────────────────┘  │
│                              ↓                                       │
│  ┌──────────────────── MLP Block ───────────────────────────────┐  │
│  │                                                               │  │
│  │  Gate Proj (GEMM⑦) ──▸ SiLU ──┐                             │  │
│  │  Up Proj   (GEMM⑧) ───────────┤──▸ element-wise mul          │  │
│  │                                └──▸ Down Proj (GEMM⑨)        │  │
│  └───────────────────────────────────────────────────────────────┘  │
│                                                                     │
│  共 9 个 GEMM 操作 / 每层                                          │
└─────────────────────────────────────────────────────────────────────┘
```

### 关键区别：两类矩阵乘法

| 类型 | 操作 | 矩阵形状 | 特点 |
|------|------|----------|------|
| **投影 GEMM** | Q/K/V/O/Gate/Up/Down proj | `[tokens, hidden] × [hidden, proj]` | 大矩阵, 标准 GEMM |
| **注意力 GEMM** | Q*Kᵀ, attn*V | `[seq, head_dim] × [head_dim, seq]` | 小 K 维度, 需 softmax 融合 |

> **投影 GEMM 占了 ~70% 的 FLOPS**，注意力 GEMM 在长序列时才变得重要。

## 6.2 投影层的 GEMM：Linear Layer 的实现

vLLM 的线性层（`QKVParallelLinear`, `ColumnParallelLinear`, `RowParallelLinear`）
最终都会调用一个 GEMM 分发函数。

In [ ]:
# vLLM 的线性层 GEMM 分发机制
# 源码: vllm/model_executor/layers/utils.py

# 核心分发逻辑（简化版）:
def dispatch_unquantized_gemm():
    """根据平台选择 GEMM 实现"""
    if current_platform.is_rocm():
        return rocm_unquantized_gemm    # AMD: hipBLASLt 或 aiter Triton GEMM
    elif current_platform.is_cpu():
        return cpu_unquantized_gemm     # CPU: MKL/BLAS
    else:
        return default_unquantized_gemm # NVIDIA: cuBLAS

def default_unquantized_gemm(layer, x, weight, bias=None):
    """NVIDIA 默认路径 —— 直接调用 cuBLAS"""
    return torch.nn.functional.linear(x, weight, bias)
    # ↑ 这最终调用 cuBLAS 的 cublasGemmEx / cublasLtMatmul

# 注意: torch.nn.functional.linear(x, W, b) 等价于:
#   output = x @ W.T + b
# PyTorch 内部会根据矩阵大小选择最优的 cuBLAS 算法

print("投影层 GEMM 默认使用 cuBLAS (通过 torch.nn.functional.linear)")
print("这是因为 cuBLAS 对标准 GEMM 有极其成熟的优化")

### 为什么投影层默认用 cuBLAS 而不是 Triton？

```
cuBLAS 优势:
├── 经过 NVIDIA 工程师数十年手动优化
├── 针对每种 GPU 架构有专门的 kernel 实现
├── 自动选择最优的 tile 大小、warp 分配
├── 支持 FP16 Tensor Core、TF32、FP64 等所有精度
└── 对 [M, N, K] 的各种组合都有调优

Triton GEMM 的定位:
├── 当需要 **融合额外操作** 时 (量化、路由、激活)
├── 当 cuBLAS 不支持的特殊矩阵格式时
├── 当需要 **自定义内存访问模式** 时 (如 MoE)
└── 在 AMD GPU 上，对小矩阵 Triton 有时更优
```

> **生活类比**: cuBLAS 像工厂流水线——标准化产品效率极高。
> Triton 像手工作坊——灵活，能做定制品，但标准品不一定更快。

### 量化模式下的投影层 GEMM

当模型使用量化（FP8, INT8, INT4）时，vLLM 使用专门的 GEMM 实现：

In [ ]:
# vLLM 量化 GEMM 的分发路径
# 源码: vllm/model_executor/layers/linear.py

# UnquantizedLinearMethod → torch.nn.functional.linear (cuBLAS)
# Fp8LinearMethod         → cutlass_fp8_gemm / cuBLAS fp8
# GPTQMarlinLinearMethod   → Marlin kernel (高度优化的 INT4 GEMM)
# AWQLinearMethod          → AWQ dequant + cuBLAS
# CompressedTensorsMethod  → 视格式而定

WEIGHT_LOADER_V2_SUPPORTED = [
    "UnquantizedLinearMethod",     # cuBLAS
    "Fp8LinearMethod",             # CUTLASS FP8 或 cuBLAS FP8
    "GPTQMarlinLinearMethod",      # Marlin CUDA kernel
    "AWQMarlinLinearMethod",       # Marlin CUDA kernel
    "CompressedTensorsLinearMethod", # 多种后端
    "FBGEMMFp8LinearMethod",       # FBGEMM 库
    "ModelOptFp8LinearMethod",     # TensorRT-LLM 的 FP8
    "ModelOptNvFp4LinearMethod",   # NV FP4
    # ... 更多量化方法
]

print("量化 GEMM 实现:")
print("  FP8  → CUTLASS (针对 Hopper/Ada 架构优化)")
print("  INT4 → Marlin kernel (CUDA, 极度手动优化)")
print("  INT8 → cuBLAS INT8 或 CUTLASS")
print("  FP4  → NVIDIA 专用 kernel")

## 6.3 注意力中的 GEMM：Q*K 和 attn*V

注意力中的矩阵乘法与投影层完全不同：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

关键区别：
- **K 维度很小** (head_dim = 64~128)，不是典型的大 GEMM
- **Q*Kᵀ 和 softmax 和 attn*V 必须融合**，否则中间张量 `[seq_len, seq_len]` 爆内存
- 这就是 **FlashAttention** 的核心思想

### vLLM 的注意力 GEMM 后端

| 后端 | Q*Kᵀ 实现 | attn*V 实现 | 适用场景 |
|------|-----------|-------------|---------|
| **FlashAttention** | CUDA (手写 kernel) | CUDA (手写 kernel) | Prefill + Decode (默认) |
| **FlashInfer** | CUDA kernel | CUDA kernel | MLA 解码, 高性能 |
| **Triton Attention** | `tl.dot(q, k)` | `tl.dot(p, v)` | Prefill + Decode (纯 Triton) |
| **Triton MLA** | `tl.dot(q, k)` + DPE 分离 | `tl.dot(p, v)` | MLA 解码 (DeepSeek) |
| **FlashMLA** | CUDA kernel | CUDA kernel | MLA 专用高性能 |
| **CUTLASS MLA** | CUTLASS | CUTLASS | MLA on Hopper |

In [ ]:
# vLLM Triton 注意力中的矩阵乘法
# 源码: vllm/v1/attention/ops/triton_prefill_attention.py

# Prefill 阶段 (FlashAttention 风格的 Triton 实现):
# 核心循环 (简化版):

"""
@triton.jit
def _fwd_kernel(Q, K, V, ...):
    # 加载 Q block: [BLOCK_M, head_dim]
    q = tl.load(Q + off_q)

    for start_n in range(0, seq_len, BLOCK_N):
        # 加载 K block: [head_dim, BLOCK_N]  (转置后)
        k = tl.load(K + off_k)

        # ===== GEMM④: Q * K^T =====
        # [BLOCK_M, head_dim] × [head_dim, BLOCK_N] → [BLOCK_M, BLOCK_N]
        qk = tl.dot(q, k)           # ← 这就是注意力中的矩阵乘法!
        qk = qk * sm_scale

        # Online softmax (FlashAttention 的核心)
        m_ij = tl.maximum(m_i, tl.max(qk, 1))
        p = tl.math.exp2(qk - m_ij[:, None])

        # 加载 V block: [BLOCK_N, head_dim]
        v = tl.load(V + off_v)

        # ===== GEMM⑤: attn * V =====
        # [BLOCK_M, BLOCK_N] × [BLOCK_N, head_dim] → [BLOCK_M, head_dim]
        acc = tl.dot(p, v, acc)      # ← 注意力加权求和
"""

print("Triton Prefill Attention 的两个 tl.dot:")
print("  1. tl.dot(q, k)  → Q*K^T  [BLOCK_M, d] × [d, BLOCK_N]")
print("  2. tl.dot(p, v)  → attn*V [BLOCK_M, BLOCK_N] × [BLOCK_N, d]")
print()
print("注意: head_dim 通常 = 64 或 128, 远小于投影 GEMM 的 K 维度")
print("这意味着 tl.dot 一次就能覆盖整个 head_dim, 不需要 K-loop")

## 6.4 GQA (Grouped Query Attention) 的矩阵乘法

GQA 是 Llama 2/3, Mistral, Qwen 等模型使用的注意力变体：
- **多个 Q head 共享同一组 K/V head**
- 例如 Llama-70B: 64 个 Q head, 8 个 KV head → `kv_group_num = 8`

### GQA 对矩阵乘法的影响

$$\text{GQA}: Q_i K_{\lfloor i/g \rfloor}^T V_{\lfloor i/g \rfloor}, \quad g = \text{kv\_group\_num}$$

In [ ]:
# GQA 在 vLLM 中的 Triton 实现
# 源码: vllm/v1/attention/ops/triton_decode_attention.py

# === 方案 1: 逐 head 处理 (标准 GQA) ===
# _fwd_kernel_stage1 中:
"""
cur_kv_head = cur_head // kv_group_num
# ↑ 多个 Q head 映射到同一个 KV head
# 每个 program 处理 1 个 Q head 的 Q*K 和 attn*V
qk = tl.sum(q[None, :] * k, 1)  # 向量内积, 非 tl.dot
acc += tl.sum(p[:, None] * v, 0)
"""

# === 方案 2: 分组批处理 (grouped kernel) ===
# _fwd_grouped_kernel_stage1 中:
"""
# BLOCK_H 个 Q head 同时处理, 共享同一 KV head
cur_head = cur_head_id * VALID_BLOCK_H + tl.arange(0, BLOCK_H)
# 多个 Q head 一起做矩阵乘法:
qk = tl.dot(q, k.to(q.dtype))       # [BLOCK_H, BLOCK_DMODEL] × [BLOCK_DMODEL, BLOCK_N]
acc += tl.dot(p.to(v.dtype), v)      # [BLOCK_H, BLOCK_N] × [BLOCK_N, BLOCK_DV]
"""

print("GQA 的两种 Triton 实现:")
print()
print("方案 1 (标准): 每个 program = 1 个 Q head")
print("  Q*K = tl.sum(q * k, axis=-1)  # 向量点积, 非矩阵乘法")
print("  适用于: kv_group_num 小 (如 Llama-7B 的 1)")
print()
print("方案 2 (分组): 每个 program = BLOCK_H 个 Q head")
print("  Q*K = tl.dot(q, k)  # 真正的矩阵乘法, 利用 Tensor Core")
print("  适用于: kv_group_num 大 (如 Llama-70B 的 8)")
print("  优势: 将 GQA 转化为 GEMM, 提升硬件利用率")

### GQA Grouped Kernel 的核心思想

```
标准 GQA (每个 head 独立):
  Q head 0  ─▸ K head 0 ─▸ Q₀*K₀ᵀ  (向量内积)
  Q head 1  ─▸ K head 0 ─▸ Q₁*K₀ᵀ  (向量内积)
  ...
  Q head 7  ─▸ K head 0 ─▸ Q₇*K₀ᵀ  (向量内积)
  ↑ 8 次 kernel launch, 每次只做向量运算

分组 GQA (BLOCK_H 个 head 一起):
  ┌ Q head 0 ┐
  │ Q head 1 │
  │ ...      │ ─▸ K head 0 ─▸ Q[0:8]*K₀ᵀ  (矩阵乘法!)
  │ Q head 7 │
  └──────────┘
  ↑ 1 次 tl.dot, 利用 Tensor Core 的 [M,K]×[K,N] 并行
```

> **生活类比**: 标准 GQA 像是 8 个人轮流查字典（同一本字典），
> 分组 GQA 像是 8 个人同时看这本字典的同一页——批量处理效率更高。

## 6.5 MLA (Multi-head Latent Attention) 的矩阵乘法

MLA 是 DeepSeek-V2/V3 的核心创新，**用低秩投影压缩 KV Cache**。

### MLA 的矩阵乘法比 GQA 多得多

```
GQA 的 GEMM:                  MLA 的 GEMM:
├── Q_proj (GEMM)             ├── h → q_c      (GEMM: W_DQ, 低秩压缩)
├── K_proj (GEMM)             ├── q_c → q_nope (GEMM: W_UQ, 解压缩)
├── V_proj (GEMM)             ├── q_c → q_pe   (GEMM: W_QR, rope 投影)
├── Q*K^T (attention)         ├── h → kv_c     (GEMM: W_DKV, KV 压缩)
├── attn*V (attention)        ├── h → k_pe     (GEMM: W_KR, rope 投影)
└── O_proj (GEMM)             ├── kv_c → k_nope (GEMM: W_UK, 解压缩) *compute path
                              ├── kv_c → v      (GEMM: W_UV, 解压缩) *compute path
                              ├── Q*K^T          (attention)
                              ├── attn*V         (attention)
                              └── O_proj         (GEMM)
```

In [ ]:
# MLA 的两条计算路径
# 源码: vllm/model_executor/layers/attention/mla_attention.py

# ===== Compute-Friendly 路径 (Prefill) =====
# 先解压 KV, 再做标准 MHA
"""
kv_c = h_t @ W_DKV                              # GEMM: [tokens, H] × [H, Lkv]
k_nope = (kv_c @ W_UK).view(Skv, N, P)          # GEMM: [Skv, Lkv] × [Lkv, N*P]
v = (kv_c @ W_UV).view(Skv, N, V)               # GEMM: [Skv, Lkv] × [Lkv, N*V]
# 然后做标准 MHA:
attn_out = flash_attention(Q, K, V)               # FlashAttention
"""

# ===== Data-Movement Friendly 路径 (Decode) =====
# 把解压权重吸收到 Q 中, 做 MQA
"""
ql_nope = einsum("snh,lnh->snl", q_nope, W_UK)   # GEMM: 吸收 W_UK 到 Q
# 然后做 MQA (不需要解压 KV!):
# Q*K 的 K 维度 = Lkv + R (如 512+64=576)
attn_out = flash_attention(
    Q=[ql_nope, q_pe],     # [tokens, N, Lkv+R]
    K=[kv_c, k_pe],        # [Skv, 1, Lkv+R]   ← MQA, 只有 1 个 KV head
    V=kv_c                 # [Skv, 1, Lkv]
)
o = einsum("snl,lnv->snv", attn_out, W_UV)       # GEMM: 后解压
"""

print("MLA 的矩阵乘法特点:")
print()
print("Decode 路径 (data-movement friendly):")
print("  1. q_nope @ W_UK  → 吸收权重到 Q (额外 GEMM)")
print("  2. Q*K^T 的 head_dim = 576 (不是标准的 128)")
print("     → kv_lora_rank(512) + rope_dim(64)")
print("  3. attn*V 后再 @ W_UV → 后解压 (额外 GEMM)")
print()
print("Prefill 路径 (compute friendly):")
print("  1. kv_c @ W_UK → 解压 K (大 GEMM)")
print("  2. kv_c @ W_UV → 解压 V (大 GEMM)")
print("  3. 标准 MHA → FlashAttention (head_dim=192)")
print()
print("DSV3 数据: Lkv=512, P=128, R=64, V=128, N=128 heads")

### MLA Decode 的 Triton 注意力 kernel

vLLM 为 MLA 提供了专门的 Triton decode kernel，最大的区别是 **BLOCK_DPE 参数**：

```
标准 GQA decode:
  qk = tl.dot(q, k)                     # Q 和 K 使用相同的 head_dim

MLA decode:
  qk  = tl.dot(q, k)                    # q_nope * k_nope, head_dim=512
  qk += tl.dot(qpe, kpe)                # q_pe * k_pe, 额外 rope dim=64
  ↑ 两次 tl.dot, 分别处理 nope 和 rope 部分
  ↑ 因为 rope 只应用于 k_pe, 不应用于 kv_c
```

这就是 `BLOCK_DPE` 参数的意义：它分离了 rope 和 non-rope 维度的计算。

## 6.6 GDN (GatedDeltaNet) 的矩阵乘法

GDN 是 Qwen3.5 等模型引入的 **线性注意力** 变体，不使用标准的 softmax attention。

In [ ]:
# GDN 的计算流程
# 源码: vllm/model_executor/layers/kda.py
# Triton 实现: vllm/model_executor/layers/fla/ops/

# GDN 不是传统的 Q*K^T softmax V
# 而是基于 Delta Rule 的循环更新:
"""
GatedDeltaNet 的核心:
  S_t = α * S_{t-1} + β_t * k_t * (v_t - kᵀ_t * S_{t-1})
  o_t = qᵀ_t * S_t

  其中 S 是一个 [d_k, d_v] 的状态矩阵 (类似 KV cache)
  核心矩阵运算:
    1. k_t * v_t^T    → 外积, 更新 state
    2. k_t^T * S      → 矩阵-向量乘, 读取旧 state
    3. q_t^T * S      → 矩阵-向量乘, 输出
"""

# FLA 库的 Triton 实现使用 chunk-wise 并行:
# 源码: vllm/model_executor/layers/fla/ops/chunk_delta_h.py
#
# chunk_delta_h: 计算 delta rule 的 hidden state 更新
# chunk_o:       计算 chunk 内的输出
# chunk_scaled_dot_kkt: 计算 K*K^T 的 scaled dot product
# wy_fast:       计算 WY 表示 (用于高效 delta rule)

print("GDN 与标准 Attention 的矩阵乘法对比:")
print()
print("标准 Attention:")
print("  Q*K^T → softmax → attn*V")
print("  复杂度: O(n²d)")
print()
print("GDN (GatedDeltaNet):")
print("  Chunk-wise 并行: 把序列分成 chunks")
print("  Chunk 内: Q*K^T (小矩阵) + delta state 更新")
print("  Chunk 间: state 递推 (隐式矩阵乘法)")
print("  复杂度: O(nCd), C=chunk_size")
print()
print("FLA 库用 Triton 实现了高效的 chunk 并行:")
print("  tl.dot 用于 chunk 内的 QK 和 state 更新")
print("  循环用于 chunk 间的 state 传递")

## 6.7 MLP 与 MoE 的矩阵乘法 — vLLM 最复杂的 Triton GEMM

**Fused MoE (Mixture of Experts)** 是 vLLM 中最复杂的 Triton 矩阵乘法实现。

### 标准 MLP 的 GEMM

```
标准 MLP (非 MoE):
  Gate = hidden @ W_gate     # GEMM⑦: cuBLAS
  Up   = hidden @ W_up       # GEMM⑧: cuBLAS
  Down = SiLU(Gate) * Up @ W_down  # GEMM⑨: cuBLAS
  ↑ 三次独立的 cuBLAS 调用
```

### MoE 的 GEMM — 为什么需要 Triton？

```
MoE:
  1. Router: 每个 token 选择 top-k 个 experts
  2. Gate = hidden[routed] @ Expert_gate[expert_id]   # 不同 token → 不同 expert 权重!
  3. Up   = hidden[routed] @ Expert_up[expert_id]
  4. Down = SiLU(Gate) * Up @ Expert_down[expert_id]

  问题: cuBLAS 不支持 "每行选不同的权重矩阵" 这种操作!
  → 需要自定义 Triton kernel
```

In [ ]:
# vLLM Fused MoE Triton Kernel — 最复杂的 Triton GEMM
# 源码: vllm/model_executor/layers/fused_moe/fused_moe.py

# fused_moe_kernel 的参数列表 (展示其复杂度):
"""
@triton.jit
def fused_moe_kernel(
    a_ptr, b_ptr, c_ptr,       # 输入/权重/输出
    b_bias_ptr,                 # 可选 bias
    a_scale_ptr, b_scale_ptr,   # 量化 scale
    topk_weights_ptr,            # Router 权重
    sorted_token_ids_ptr,        # Token 路由映射
    expert_ids_ptr,              # Expert 索引
    num_tokens_post_padded_ptr,  # 填充后 token 数
    N, K, EM,                    # 矩阵维度
    ...
    # 量化参数
    group_n, group_k,            # Block-wise 量化
    use_fp8_w8a8, use_int8_w8a8, use_int8_w8a16,
    per_channel_quant, HAS_BIAS,
    # GEMM 参数
    BLOCK_SIZE_M, BLOCK_SIZE_N, BLOCK_SIZE_K,
    GROUP_SIZE_M, SPLIT_K,
    MUL_ROUTED_WEIGHT,
):
"""

print("Fused MoE Kernel 相比标准 GEMM 的额外操作:")
print()
print("1. Token 路由 (sorted_token_ids):")
print("   每个 block 从 sorted_token_ids 查找真实 token 位置")
print("   → a_ptrs 不是连续的, 需要 indirect load")
print()
print("2. Expert 权重选择 (expert_ids):")
print("   每个 M-block 可能对应不同的 expert")
print("   → b_ptrs = b_ptr + off_experts * stride_be")
print("   → 权重矩阵的基地址随 block 变化")
print()
print("3. 量化反缩放 (FP8/INT8):")
print("   支持 tensor-wise, channel-wise, block-wise 三种粒度")
print("   → 在累加完成后乘 scale: acc *= a_scale * b_scale")
print()
print("4. Router 权重加权 (MUL_ROUTED_WEIGHT):")
print("   acc *= moe_weight  # 在最终存储前乘路由权重")
print()
print("5. Swizzle (GROUP_SIZE_M):")
print("   L2 Cache 优化的 block 调度 (与 Ch.13 相同)")

### Fused MoE Kernel 的核心 GEMM 循环

下面是 `fused_moe_kernel` 中矩阵乘法的核心代码：

In [ ]:
# Fused MoE Kernel 的核心 GEMM 循环 (简化)
# 源码: vllm/model_executor/layers/fused_moe/fused_moe.py:498-535

"""
# FP32 累加器 (与 Ch.16 混合精度策略相同)
accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
    # 加载 A block (输入 token features)
    a = tl.load(a_ptrs,
                mask=token_mask[:, None] & (offs_k[None, :] < K - k * BLOCK_SIZE_K),
                other=0.0)
    # 加载 B block (expert 权重)
    b = tl.load(b_ptrs,
                mask=offs_k[:, None] < K - k * BLOCK_SIZE_K,
                other=0.0)

    # ==== GEMM 核心 ====
    if use_int8_w8a16:
        accumulator = tl.dot(a, b.to(compute_type), acc=accumulator)
    elif use_fp8_w8a8 or use_int8_w8a8:
        if group_k > 0 and group_n > 0:
            # Block-wise 量化: 每个 block 有独立的 scale
            a_scale = tl.load(a_scale_ptrs + offs_ks * stride_ask)
            b_scale = tl.load(b_scale_ptrs + offs_ks * stride_bsk)
            accumulator += tl.dot(a, b) * a_scale[:, None] * b_scale[None, :]
        else:
            # FP8 fast accumulate
            accumulator = tl.dot(a, b, acc=accumulator)
    else:
        # 标准 FP16/BF16 路径
        accumulator += tl.dot(a, b)  # ← 与 Ch.18 终极 GEMM 结构完全相同!

    # 推进 K 维度指针
    a_ptrs += BLOCK_SIZE_K * stride_ak
    b_ptrs += BLOCK_SIZE_K * stride_bk
"""

print("核心 tl.dot 与标准 GEMM 完全相同!")
print("区别在于外层的 token 路由、expert 选择、量化处理")
print()
print("GEMM 循环结构对比:")
print("  标准 GEMM:    for k: acc += tl.dot(a, b)")
print("  Fused MoE:    for k: acc += tl.dot(a, b)  ← 相同!")
print("  + expert routing  (a_ptr 间接寻址)")
print("  + weight select   (b_ptr 按 expert_id 偏移)")
print("  + quant scale     (acc *= scale)")
print("  + router weight   (acc *= moe_weight)")

### GPT-OSS Triton Kernels: 更高性能的 MoE GEMM

vLLM 还集成了来自 NVIDIA GPT-OSS 的高性能 MoE kernel:

```
# 源码: vllm/model_executor/layers/fused_moe/gpt_oss_triton_kernels_moe.py

from triton_kernels.matmul_ogs import matmul_ogs  # NVIDIA 官方 Triton kernel

# matmul_ogs = "Matrix Multiply with Output Gather/Scatter"
# 特点:
# 1. 使用 Bitmatrix 编码 token-expert 路由 (比 sorted_token_ids 更紧凑)
# 2. 支持 FusedActivation (SwiGLU 等在 GEMM 内部融合)
# 3. 使用 SparseMatrix 进行稀疏路由
# 4. 经过 NVIDIA 团队深度优化
```

## 6.8 算子总览表

| 场景 | 矩阵形状 | 默认实现 | Triton 实现 | 最优选择 |
|------|----------|---------|------------|---------|
| **Q/K/V/O 投影** | `[T, H] × [H, D]` | cuBLAS | ROCm aiter | cuBLAS (NVIDIA) |
| **MLP Gate/Up/Down** | `[T, H] × [H, D]` | cuBLAS | — | cuBLAS |
| **MoE Gate/Up/Down** | `[T_routed, H] × [E, H, D]` | — | `fused_moe_kernel` | Triton (专用) |
| **Prefill Q*Kᵀ** | `[BLOCK_M, d] × [d, BLOCK_N]` | FlashAttn (CUDA) | `tl.dot(q, k)` | FlashAttn |
| **Prefill attn*V** | `[BLOCK_M, BLOCK_N] × [BLOCK_N, d]` | FlashAttn (CUDA) | `tl.dot(p, v)` | FlashAttn |
| **Decode Q*Kᵀ** | `[1, d] × [d, seq]` | FlashAttn/FlashInfer | `tl.sum(q * k)` | FlashInfer |
| **GQA Decode** | `[BLOCK_H, d] × [d, BLOCK_N]` | FlashAttn | grouped kernel | FlashAttn |
| **MLA Decode** | `[BLOCK_H, 512+64] × [576, BLOCK_N]` | FlashMLA/FlashInfer | `tl.dot` × 2 | FlashMLA |
| **MLA Prefill (解压)** | `[Skv, 512] × [512, N*128]` | cuBLAS | — | cuBLAS |
| **GDN chunk** | `[C, d_k] × [d_k, d_v]` | — | FLA Triton | FLA Triton |
| **FP8 投影** | `[T, H] × [H, D]` | CUTLASS | — | CUTLASS |
| **INT4 投影** | `[T, H] × [H, D]` | Marlin CUDA | — | Marlin |

## 6.9 性能最关键的矩阵乘法

### 哪些 GEMM 是瓶颈？

对于不同的推理阶段和模型架构:

**Prefill 阶段 (长 prompt)**:
- 投影 GEMM 是主要瓶颈 (cuBLAS 已高度优化)
- 注意力 GEMM 随 seq_len² 增长, FlashAttention 至关重要

**Decode 阶段 (逐 token 生成)**:
- 投影 GEMM 退化为 GEMV (batch_size=1), memory-bound
- 注意力的 KV cache 读取成为瓶颈
- MoE 模型中, fused_moe_kernel 是热点

**MoE 模型 (Mixtral, DeepSeek-V3)**:
- `fused_moe_kernel` 是计算密集型瓶颈
- 每个 token 激活 top-k experts, 每个 expert 有独立的 GEMM
- 这是 **vLLM 中唯一必须用 Triton 实现的 GEMM** (cuBLAS 无法处理路由逻辑)

**MLA 模型 (DeepSeek-V2/V3)**:
- 额外的压缩/解压 GEMM
- Decode 时 head_dim=576 的大注意力 GEMM

### 下一章预告

> Notebook 07 将深入 `fused_moe_kernel` — vLLM 中最复杂的 Triton GEMM，
> 从零实现并与 cuBLAS 对比性能。

## 6.10 源码映射表

| 本章内容 | vLLM 源码位置 | 说明 |
|----------|--------------|------|
| 线性层 GEMM 分发 | `vllm/model_executor/layers/utils.py:298` | `dispatch_unquantized_gemm()` |
| 默认 GEMM | `vllm/model_executor/layers/utils.py:113` | `default_unquantized_gemm()` → cuBLAS |
| 线性层基类 | `vllm/model_executor/layers/linear.py` | `ColumnParallelLinear` 等 |
| Triton Prefill Attention | `vllm/v1/attention/ops/triton_prefill_attention.py:37` | `_fwd_kernel` |
| Triton Decode Attention | `vllm/v1/attention/ops/triton_decode_attention.py:59` | `_fwd_kernel_stage1` |
| GQA Grouped Decode | `vllm/v1/attention/ops/triton_decode_attention.py:248` | `_fwd_grouped_kernel_stage1` |
| MLA Wrapper | `vllm/model_executor/layers/mla.py:33` | `MultiHeadLatentAttentionWrapper` |
| MLA Attention | `vllm/model_executor/layers/attention/mla_attention.py` | 两条计算路径 |
| MLA Triton Backend | `vllm/v1/attention/backends/mla/triton_mla.py` | Triton MLA decode |
| GDN Attention | `vllm/v1/attention/backends/gdn_attn.py` | GatedDeltaNet |
| GDN Triton Ops | `vllm/model_executor/layers/fla/ops/` | chunk_delta_h, chunk_o 等 |
| Fused MoE Kernel | `vllm/model_executor/layers/fused_moe/fused_moe.py:315` | `fused_moe_kernel` |
| Batched MoE | `vllm/model_executor/layers/fused_moe/fused_batched_moe.py:39` | `moe_mmk` |
| GPT-OSS MoE | `vllm/model_executor/layers/fused_moe/gpt_oss_triton_kernels_moe.py` | `matmul_ogs` |